<a href="https://colab.research.google.com/github/Dishitha09/csda-al/blob/main/notebooks/CSDA_AL_ISIC2019_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSDA-AL: Class-Sensitive Deep Active Learning on ISIC 2019
### Master Diagnosis, Next-Stage Controlled Experiments (A0–A13), and Final Validation

This notebook provides the complete experimental pipeline for **CSDA-AL** on the **ISIC 2019 dermoscopy dataset** (25,331 images, 8 classes, 53.9:1 class imbalance ratio) under strictly identical canonical splits and budget parity.

## Step 1: Setup Environment & Clone Repository
Resets to clean `/content` and clones the latest codebase from GitHub.

In [ ]:
import os
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

# Reset to /content and cleanly fetch latest repository
%cd /content
if os.path.exists("/content/csda-al"):
    !rm -rf /content/csda-al

!git clone https://github.com/Dishitha09/csda-al.git
%cd /content/csda-al

!pip install -q -e .
!pip install -q scikit-learn pandas matplotlib
print("✓ Repository ready at:", os.getcwd())

## Step 2: Technical Audit & Root-Cause Diagnosis (`--mode diagnose`)
Audits dataset class distribution, reveals the **Argmax Quota Trap**, **Loss Weight Singularity**, and **Cold-Start Density Bias**.

In [ ]:
!python run_isic_pipeline.py --mode diagnose

## Step 3: Run Baseline vs Improved CSDA-AL Across 3 Seeds (`--mode improve`)
Executes Random vs Improved CSDA-AL across seeds `42, 43, 44` with bounded class-balanced loss and soft-target deficit matching.

In [ ]:
!python run_isic_pipeline.py --mode improve --seeds 42 43 44

## Step 4: Run Initial Ablation Suite A0–A8 (`--mode ablation`)
Executes the baseline ablation matrix (A0 through A8) across seeds 42, 43, 44.

In [ ]:
!python run_isic_pipeline.py --mode ablation --seeds 42 43 44

## Step 5: Execute Next-Stage Controlled Experiments A9–A13 (`--mode next_stage`)
Executes the targeted next-stage experiments:
- **A9**: Class-Weighted Training + Calibrated Cold-Start
- **A10**: Class-Weighted Training + Uncertainty Acquisition
- **A11**: Class-Weighted + Uncertainty + Revised Cosine Diversity
- **A12**: Validation-Tuned CSDA-AL Weights Grid Search
- **A13**: Final Integrated CSDA-AL on Locked Test Set

In [ ]:
!python run_isic_pipeline.py --mode next_stage --seeds 42 43 44

## Step 6: Display Master Benchmark Comparison Matrix (A0 to A13)
Displays all 14 evaluated configurations across seeds 42, 43, 44 with Mean ± Std.

In [ ]:
import pandas as pd
from IPython.display import display

df_master = pd.read_csv("outputs/isic2019/metrics/next_stage_benchmark_comparison.csv")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
display(df_master)

## Step 7: Display Publication Figures Inline
Visualizes confusion matrices, learning curves, and per-class diagnostic recovery.

In [ ]:
from IPython.display import Image, display

print("Figure 10: Confusion Matrix Comparison (Baseline vs Proposed CSDA-AL C4)")
display(Image("results/figures/fig10_confusion_matrix_comparison.png", width=850))

print("Figure 6: Macro-F1 Score vs Annotation Budget")
display(Image("results/figures/fig6_macro_f1_vs_budget.png", width=750))

print("Figure 11: Per-Class F1-Score (Eliminating Minority Diagnostic Collapse)")
display(Image("results/figures/fig11_per_class_performance.png", width=800))